# Pattern 1: Reflection

Reflection makes the agent reflect on its output. Or more generally, it makes multiple LLMs talk to each other in something like a **peer review process**. The reflection agent suggests modifications, additions, improvements in the writing style, and so on. This iterative process[^iterative] often leads to substantial gains in output quality, as the generation model benefits from external critique (i.e. different model, or just a different execution process[^reflection]). Reflection is ideal for high-stakes, critical tasks where reflection acts as a cheap QA step.

[^iterative]: The cost of running the model 2-3 times is far less than the cost of a significant error. For situations where the cost of a mistake is negligible, then using reflection isn't a good tradeoff.

[^reflection]: The reflection model is focused on evaluation, error detection, factual verification, or alignment with constraints. So it has a more specific goal than generating content from scratch.

![**Reflection Pattern**. Two LLMs iteratively improve the generated response through an iterative review process.](./img/pattern-reflective.png){#fig-pattern-reflective}

## Reflection steps

**Inference client.** Initializing the client for LLM inference and loading the API keys:

In [1]:
import pandas as pd

from openai import OpenAI
from notebooks.utils import load_dotenv, print
from IPython.display import display_markdown

load_dotenv(verbose=True)
client = OpenAI()

Loaded env variable: OPENAI_API_KEY
Loaded env variable: GROQ_API_KEY


### System prompts

We will create two separate chat histories, one for generation and another for reflection. We set the generation system prompt as a developer tasked to write high-quality Python code. On the other hand, we set the reflection system prompt such that it only responds with feedback instead of rewriting the whole thing. Finally, we instruct the reflection agent to write `APPROVED` when satisfied so we can terminate the loop.

In [2]:
STOP_WORD = "APPROVED"

BASE_GENERATION_SYSTEM_PROMPT = """
Your task is to Generate the best content possible for the user's request.
If the user provides critique, respond with a revised version of your previous attempt.
You must always output the revised content.
"""

BASE_REFLECTION_SYSTEM_PROMPT = f"""
You are tasked with generating critique and recommendations on the user's generated content. 
Your role is to help the user improve by pointing out strengths, weaknesses, and opportunities 
for refinement. 

You must NEVER provide full solutions, rewritten versions of the content, or long verbatim outputs. 
You may use short illustrative examples (1-3 lines or a single sentence) only when necessary to clarify 
a point. Providing a complete solution is a policy violation. 

If the user content has something wrong or something to be improved, output ONLY a clear list of 
recommendations and critiques. 

If you are satisfied and have no further strong recommendations, output EXACTLY the single word:

{STOP_WORD}

GUIDELINES FOR CRITIQUE:
- Forbidden Example: Rewriting the entire essay, code, or design for the user.
- Forbidden Example: Giving the full, corrected version of the user's work.
- Allowed Example: "Consider clarifying your thesis statement, e.g., make it one clear sentence."
- Good Example: Pointing out issues, suggesting improvements, or giving high-level recommendations without completing the work for the user.

GUIDELINES FOR APPROVAL:
- You must be fully satisfied with the content before approving.
- You must have checked that all past issues have been fully addressed.
- You must be sure there are no remaining issues, weaknesses, or areas for improvement.
- "{STOP_WORD}" must appear alone on a line, with no emojis, punctuation, or explanations.
- Do not mix "{STOP_WORD}" with any feedback or comments.
- Forbidden Example: "{STOP_WORD}, but consider improving your introduction."
- Good Example: "{STOP_WORD}"
"""

SHARED_DEFINITION_OF_DONE = """
DEFINITION OF DONE: The best solution is the SIMPLEST correct implementation that:
- SOLVES THE USER'S SPECIFIC PROBLEM COMPLETELY AND APPROPRIATELY
- Is readable and maintainable for the intended use case
- Avoids unnecessary complexity, over-engineering, or premature optimization  
- Uses the appropriate level of robustness (not necessarily maximal robustness)
- Prioritizes clarity and understandability over cleverness
- Delivers exactly what the user needs, nothing more and nothing less

KEY PRINCIPLE: The solution should be as simple as possible, but no simpler. 
It must address the user's actual needs while avoiding gold-plating.
"""

CODE_GENERATION_SYSTEM_PROMPT = "\n".join(["""
You are a Python programmer tasked with generating high quality Python code.
Generate exactly one Python implementation that prioritizes SIMPLICITY, READABILITY, and PRACTICALITY.
Aim for the simplest correct solution that solves the problem without over-engineering.
Avoid unnecessary complexity, clever tricks, or advanced features unless absolutely necessary.
Do not provide multiple options, explanations, or alternative approaches.
Output only the final code in a fenced Python block.
""", SHARED_DEFINITION_OF_DONE, BASE_GENERATION_SYSTEM_PROMPT])

CODE_REFLECTION_SYSTEM_PROMPT = "\n".join(["""
You are a Python programmer and strict code reviewer.

USER'S ORIGINAL REQUEST:
{user_prompt}

**Consider BOTH the user's specific needs AND our quality standards:**                                           

Your goal is to produce the simplest correct solution possible.
Avoid unnecessary complexity, clever tricks, or over-engineering.
Prioritize readability, maintainability, and clarity over novelty.

FORMAT REQUIREMENT:
- You MUST format your feedback in a **Markdown table** with the columns: | Issue | Details | Recommendation |
- Each row should contain exactly one critique and its corresponding recommendation.
- Do not use bullet points, numbered lists, or plain text for critiques — only a Markdown table.
                                           
Providing a complete solution is a policy violation. 
Forbidden Example (DO NOT DO THIS): Providing a full class or function rewrite. 
Your role is to help the user learn by giving feedback, not by coding for them. 
Allowed Example: “Consider validating input type, e.g., `if not isinstance(n, int): ...` ”

""", SHARED_DEFINITION_OF_DONE, BASE_REFLECTION_SYSTEM_PROMPT])

:::{.callout-caution}
Tuning the prompts took the most time / effort during the writing of this section. (ᵕ—ᴗ—) What worked for me: adding a **shared definition of done**, and having similar goals for both agents. In theory, having divergent goals can be good, but in practice it lead to agents going off-track, or getting into add-remove cycles. Or one agent dominating the other. Finally, the [reflection agent's system prompt]{.underline} include the **user prompt** to ground the agent's analysis in the specific context and intent of the user's request, while still maintaining the required quality standards.

:::

### Model choice

For the current task (generating code for a simple function), we use:

In [3]:
GENERATION_MODEL = "gpt-4.1"
REFLECTION_MODEL = "o3"

| Role        | Focus                                   |
|-------------|------------------------------------------------------|
| **Generation** | Creativity, fluency, diverse output. Feedback incorporation.                 |
| **Reflection** | Evaluation, error detection, factual verification, constraint alignment |


From our experiments (and performing code review IRL), reviewing is a nontrivial task: following guidelines, spotting subtle issues, and enforcing consistency needs strong reasoning capacity and attention to detail. We generally had best results with a fairly strong [reasoning model]{.underline} (e.g. `o3` and `gpt-oss`) as reflection model.

:::{.callout-tip}
## Guide to model sizes

| Generation | Reflection | Notes |
|------------|------------|--------------------------------|
| Large | Large | Simplest approach. |
| Medium | Large | Reduces generation cost. Useful when it's easier to write than critique (e.g. summarization). |
| Large | Medium | Serves as a **good baseline**. Cost-effective, but may miss subtle errors.  |
| Large | Fine-tuned, specialized | Can outperform generalists on specific feedback tasks. |

A [reasoning model]{.underline} as reflector can be considered when dealing with complex problem solving, planning, logical critique, and multi-step reasoning. For optimizing creativity and style, the reflector can be a general-purpose completion model.
:::

### Generation step

We now ask the LLM to write an implementation of the Fibonacci sequence. Since it's only used for a quick demo, we expect the agents to converge to a simple solution. Pushing user prompt to generation agent:

In [4]:
from notebooks.agents.utils import Deployment
from notebooks.agents.chat import ChatHistory, ChatCompletions

USER_PROMPT = """
Generate a Python implementation of merge sort. Don't worry about types.
This will only be used for a quick demo, so consider adding usage examples. 
"""

# Initialize deployments and histories
generator = ChatCompletions(Deployment(client, GENERATION_MODEL))
reflector = ChatCompletions(Deployment(client, REFLECTION_MODEL))
generation_chat_history = ChatHistory(CODE_GENERATION_SYSTEM_PROMPT)
reflection_chat_history = ChatHistory(CODE_REFLECTION_SYSTEM_PROMPT.format(user_prompt=USER_PROMPT))

# Initialize generation with user prompt
generation_chat_history.update(role="user", prompt=USER_PROMPT)

**Initial version.** As usual, GA has role `assistant`. We send over the response to the RA with role `user`[^role].

[^role]: Agents acting in behalf of the user hence the `user` role.

In [5]:
code = generator.create(generation_chat_history)
generation_chat_history.update(prompt=code, role="assistant")
reflection_chat_history.update(prompt=code, role="user")

:::{.callout-note collapse="false"}
## Initial generated code

In [6]:
#| echo: false
display_markdown(code, raw=True)

```python
def merge_sort(arr):
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    return merge(left, right)

def merge(left, right):
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result

# Usage examples:
arr1 = [5, 2, 9, 1, 5, 6]
arr2 = [3, -4, 0, 12, 7]
print(merge_sort(arr1))  # [1, 2, 5, 5, 6, 9]
print(merge_sort(arr2))  # [-4, 0, 3, 7, 12]
```

:::

### Reflection step

The generated critique is likewise sent over to the GA with `user` role.

In [7]:
review = reflector.create(reflection_chat_history)
reflection_chat_history.update(prompt=review, role="assistant")
generation_chat_history.update(prompt=review, role="user")

:::{.callout-note collapse="false"}
## Feedback from reflection

In [8]:
#| echo: false
display_markdown(review, raw=True)

| Issue | Details | Recommendation |
|-------|---------|----------------|
| Absent documentation | Neither function explains purpose, parameters, or return values, which may confuse readers skimming the demo code. | Add a brief docstring to `merge_sort` (and optionally `merge`) describing what the function does and that it returns a new sorted list. |
| No input validation or graceful failure | The functions assume the input is an indexable, sliceable collection of comparable elements; passing an unsupported type will raise a low-level error. | Consider a lightweight check such as `if not hasattr(arr, "__getitem__"):` to raise a clear `TypeError` with a helpful message. |
| Potentially high memory overhead from slicing | Each recursive call uses `arr[:mid]` and `arr[mid:]`, creating many intermediate lists and increasing space complexity. | Mention this trade-off in comments or docstring so users know it is a simplicity choice; if memory becomes a concern, suggest passing indices instead. |
| Recursion depth limits | Deeply nested or very large inputs could exceed Python’s recursion limit, leading to `RecursionError`. | Add a short note in the docstring (or a comment) warning about this limit and suggesting iterative alternatives for production use. |
| Missing `if __name__ == "__main__":` guard | Usage examples will run automatically if the module is imported elsewhere, which may be unintended. | Wrap the example calls inside a main guard so they execute only when the file is run as a script. |
| Unclear whether the original list is mutated | Users unfamiliar with functional style might expect in-place sorting. The function actually returns a new list and leaves the original unchanged. | State explicitly in the docstring (or an inline comment) that the original list remains unmodified; this aids user expectations. |

:::

:::{.callout-tip}
Out of all models we've tested, only OpenAI `o-` models strictly followed the review format.

:::

**Histories.** Chat histories after the first exchange. Both histories store the **generation>reflection** steps (in that causal order), only with different role assignments. This will be followed by the next generation step, and we keep iterating until the stopping condition is triggered by the RA. From the following tables we see that the generation history should have [even max length]{.underline}, while the reflection history should have [odd max length]{.underline}.

In [9]:
pd.DataFrame(generation_chat_history)

,role,content
0,system,\nYou are a Python programmer tasked with gene...
1,user,\nGenerate a Python implementation of merge so...
2,assistant,```python\ndef merge_sort(arr):\n if len(ar...
3,user,| Issue | Details | Recommendation |\n|-------...


In [10]:
pd.DataFrame(reflection_chat_history)

,role,content
0,system,\nYou are a Python programmer and strict code ...
1,user,```python\ndef merge_sort(arr):\n if len(ar...
2,assistant,| Issue | Details | Recommendation |\n|-------...


<span style="display: block; margin-bottom: 0.5em;"> </span>

Moreover, observe that we get the correct role sequence for the generation agent: `system` > `user` > [`assistant` > `user`] (one cycle). Similarly, for the refection agent, its `system` > [`user` > `assistant`] (one cycle). These are invariants even when we reach max length and the messages are replaced since messages are replaced two at a time.

## Full implementation

Combining the discussion and observations in a single class:

In [11]:
class ReflectionAgent:
    def __init__(self, 
        generator: Deployment, 
        reflector: Deployment,
        generation_system_prompt="",
        reflection_system_prompt="",
        shared_definition_of_done="",
    ):
        self.generator = ChatCompletions(generator)
        self.reflector = ChatCompletions(reflector)
        self.gen_prompt = "\n".join([generation_system_prompt, shared_definition_of_done, BASE_GENERATION_SYSTEM_PROMPT])
        self.ref_prompt = "\n".join([reflection_system_prompt, shared_definition_of_done, BASE_REFLECTION_SYSTEM_PROMPT])

    def run(self, user_prompt, max_iter=10, history_max_len=6) -> dict:
        """Iterate generation-reflection cycles until max_iter or `APPROVED` found in reflection."""
        assert history_max_len % 2 == 0, "history_max_len must be even"

        # see remarks below
        ref_prompt = self.ref_prompt.format(user_prompt=user_prompt)
        ref_history = ChatHistory(ref_prompt, max_len=history_max_len-1, fixed_n=1)
        gen_history = ChatHistory(self.gen_prompt, max_len=history_max_len, fixed_n=2)
        gen_history.update(prompt=user_prompt, role="user")

        for step in range(max_iter):

            # generate and push to ref history as user
            generation = self.generator.create(gen_history)
            gen_history.update(prompt=generation, role="assistant")
            ref_history.update(prompt=generation, role="user")

            # critique and push to gen history as user
            reflection = self.reflector.create(ref_history)
            ref_history.update(prompt=reflection, role="assistant")
            gen_history.update(prompt=reflection, role="user")

            if STOP_WORD in reflection:
                print("[Stop Sequence found. Stopping the reflection loop.]")
                break
        
        return {
            "generation": generation,
            "steps": step + 1,
            "generation_history": gen_history,
            "reflection_history": ref_history,
        }

:::{.callout-note}
Note that the user prompt is injected into the reflection prompt so that the reflection model aligns with user objectives. Then, the process starts with the user prompt pushed to the generation agent. Finally, agents on track in terms of quality standards with a shared [definition of done](https://www.atlassian.com/agile/project-management/definition-of-done). Generation history has [even]{.underline} length: 2 fixed prompts and 2 messages per iteration. On the other hand, reflection history has [odd]{.underline}, i.e. length of generation history minus 1 (only 1 fixed prompt).
:::

Running the process for a few iterations:

In [12]:
reflection_agent = ReflectionAgent(
    generator=Deployment(client, GENERATION_MODEL),
    reflector=Deployment(client, REFLECTION_MODEL),
    generation_system_prompt=CODE_GENERATION_SYSTEM_PROMPT,
    reflection_system_prompt=CODE_REFLECTION_SYSTEM_PROMPT,
    shared_definition_of_done=SHARED_DEFINITION_OF_DONE,
)

output = reflection_agent.run(user_prompt=USER_PROMPT)

[Stop Sequence found. Stopping the reflection loop.]


### Final comments

In [13]:
output["steps"]

4

Setting `history_max_len=6` means that the *last two* review-generation cycle is stored for the generation model to reference in its next generation step. Although here, it's approved so we don't go through another cycle.
Thus, (`history_max_len` - 2) / 2 is the number of past cycles the generation model can reference. Also, it's nice that the first user prompt is retained so it's like the last two generations are done *only* with the original user and system prompt in mind.

In [14]:
pd.DataFrame(output["generation_history"])

,role,content
0,system,\nYou are a Python programmer tasked with gene...
1,user,\nGenerate a Python implementation of merge so...
2,assistant,"```python\n""""""\nA simple implementation of mer..."
3,user,| Issue | Details | Recommendation |\n|-------...
4,assistant,"```python\n""""""\nA simple implementation of mer..."
5,user,APPROVED


<span style="display: block; margin-bottom: 0.5em;"> </span>


This is also the number of past cycles the reflection model references:

In [15]:
pd.DataFrame(output["reflection_history"])

,role,content
0,system,\nYou are a Python programmer and strict code ...
1,user,"```python\n""""""\nA simple implementation of mer..."
2,assistant,| Issue | Details | Recommendation |\n|-------...
3,user,"```python\n""""""\nA simple implementation of mer..."
4,assistant,APPROVED


<span style="display: block; margin-bottom: 0.5em;"> </span>


Checking out the last code version and the final critique from the reflection agent:

In [16]:
#| echo: false
display_markdown(output["reflection_history"][1]["content"], raw=True)

```python
"""
A simple implementation of merge sort in Python.

- Provides the `merge_sort` function, which returns a new sorted list from an input list.
- Demonstrates merge sort's correctness and stability with various example cases.
- Run this script directly to see merge sort in action on a range of demo data.
"""

from typing import List, Any

def merge_sort(arr: List[Any]) -> List[Any]:
    """Sorts a list using the merge sort algorithm (returns a new sorted list).
    
    Args:
        arr: List of comparable elements.
        
    Returns:
        New sorted list containing the same elements as arr.
    
    Note:
        The original list is not modified; a new sorted list is returned.
        Uses list slicing, which copies data at each recursion level.
        For large datasets, consider an in-place variant to reduce memory usage.
        Merge sort is stable: equal elements retain their original order.
        Very large lists may hit Python's maximum recursion depth.
    """
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    return merge(left, right)

def merge(left: List[Any], right: List[Any]) -> List[Any]:
    """Merges two sorted lists into one sorted list.
    
    Preconditions:
        Both `left` and `right` must be sorted lists.
    
    Args:
        left: First sorted list.
        right: Second sorted list.
        
    Returns:
        Merged sorted list containing all elements from left and right.
    """
    result = []
    i = j = 0
    # Merge sort is stable; this comparison preserves order of equal elements.
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result

if __name__ == "__main__":
    # Demo and edge cases.
    demo_cases = [
        [],
        [1],
        [5, 3, 8, 4, 2],
        [10, 7, 2, 1, 9, 5],
        [1, 2, 3, 4, 5],
        [2, 2, 1, 1, 3]
    ]
    for idx, case in enumerate(demo_cases, 1):
        print(f"Case {idx}: Original: {case}")
        sorted_case = merge_sort(case)
        print(f"Case {idx}: Sorted:   {sorted_case}\n")
```

In [17]:
#| echo: false
display_markdown(output["reflection_history"][2]["content"], raw=True)

| Issue | Details | Recommendation |
|-------|---------|---------------|
| Broad `Any` type hint weakens static checking | `List[Any]` lets any unrelated types through, yet the algorithm relies on elements being mutually comparable (`<=`). | Consider a `TypeVar` with a bound comparable protocol (e.g., `T = TypeVar("T", bound="SupportsLessThan")`) or at least mention the comparability requirement in the type comment. |
| Helper function exposed publicly | `merge` is intended for internal use but is imported when the module is reused elsewhere. | Prefix the function name with an underscore or add `__all__ = ["merge_sort"]` to signal its private nature. |
| Docstring formatting could align with PEP 257 | First line of each docstring should be a short sentence ending with a period, followed by a blank line. Current format starts with “Sorts a list...” but lacks the blank line. | Insert a blank line after the first summary line in each docstring for conventional style. |
| Redundant note in `merge_sort` docstring | The note about recursion depth and slicing is useful but somewhat long for a single “Note:” block. | Break into separate “Notes:” list or condense wording to keep docstring concise while retaining key caveats. |
| Demo lacks failure illustration | Only prints successful sorts; users won’t see what happens with unsortable mixed types or extremely long inputs. | Add a short comment suggesting how to experiment with mixed types or large lists to observe limitations instead of executing such cases by default.

The review actually makes sense. Moreover, the generation agent followed the recommendations resulting in an approval. See the diff and the final version below.

### Final output

Comparing the results to see the effect of reflection:

:::{.callout-note collapse="false"}
## Final approved output

In [18]:
#| echo: false
display_markdown(output["generation"], raw=True)

```python
"""
A simple implementation of merge sort in Python.

- Provides the `merge_sort` function, which returns a new sorted list from an input list.
- Demonstrates merge sort's correctness and stability with various example cases.
- Run this script directly to see merge sort in action on a range of demo data.

Notes:
- Elements in the input list must be mutually comparable (supporting the <= operator).
- The original list is not modified; a new sorted list is returned.
- On very large inputs, recursion or memory limits may be reached due to Python's recursion depth and slicing.
"""

from typing import List, TypeVar

T = TypeVar("T")  # Elements must support <= comparisons with each other

__all__ = ["merge_sort"]

def merge_sort(arr: List[T]) -> List[T]:
    """Return a new sorted list from the input list using merge sort.

    The original input list is not modified.

    Args:
        arr: List of mutually comparable elements.

    Returns:
        New sorted list containing the same elements as arr.
    """
    if len(arr) <= 1:
        return arr
    mid = len(arr) // 2
    left = merge_sort(arr[:mid])
    right = merge_sort(arr[mid:])
    return _merge(left, right)

def _merge(left: List[T], right: List[T]) -> List[T]:
    """Merge two sorted lists into one sorted list.

    Both `left` and `right` must be sorted lists of mutually comparable elements.

    Args:
        left: First sorted list.
        right: Second sorted list.

    Returns:
        Merged sorted list containing all elements from left and right.
    """
    result = []
    i = j = 0
    while i < len(left) and j < len(right):
        if left[i] <= right[j]:
            result.append(left[i])
            i += 1
        else:
            result.append(right[j])
            j += 1
    result.extend(left[i:])
    result.extend(right[j:])
    return result

if __name__ == "__main__":
    # Demo and edge cases.
    demo_cases = [
        [],
        [1],
        [5, 3, 8, 4, 2],
        [10, 7, 2, 1, 9, 5],
        [1, 2, 3, 4, 5],
        [2, 2, 1, 1, 3]
    ]
    for idx, case in enumerate(demo_cases, 1):
        print(f"Case {idx}: Original: {case}")
        sorted_case = merge_sort(case)
        print(f"Case {idx}: Sorted:   {sorted_case}\n")

    # To observe error cases, try passing a list with elements of mixed, incomparable types
    # (e.g., [1, "a", 2]). This will raise a TypeError at runtime.
    # To observe recursion limits, try very large lists (e.g., merge_sort(list(range(10**5)))). 
    # These are not shown here to keep the demo simple.
```

:::

In [19]:
#| echo: false
import difflib
from IPython.display import display_html

text1 = output["reflection_history"][1]["content"]
text2 = output["reflection_history"][3]["content"]
lines1 = text1.splitlines(keepends=True)
lines2 = text2.splitlines(keepends=True)

differ = difflib.HtmlDiff()
html_diff = differ.make_file(lines1, lines2, fromdesc="Original Text", todesc="Modified Text")
display_html(html_diff, raw=True)

<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN"
 "http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">

 

 
 
 
 
 

 
 
 
 
 
 Original Text Modified Text 
 
 f 1 ```python f 1 ```python 
 2 """ 2 """ 
 3 A simple implementation of merge sort in Python. 3 A simple implementation of merge sort in Python. 
 4 4 
 5 - Provides the `merge_sort` function, which returns a new sorted list from an input list. 5 - Provides the `merge_sort` function, which returns a new sorted list from an input list. 
 6 - Demonstrates merge sort's correctness and stability with various example cases. 6 - Demonstrates merge sort's correctness and stability with various example cases. 
 7 - Run this script directly to see merge sort in action on a range of demo data. 7 - Run this script directly to see merge sort in action on a range of demo data. 
 n n 8   
 9 Notes: 
 10 - Elements in the input list must be mutually comparable (supporting the <= operator). 
 11 - The original list is not modified; a new sorted list is returned. 
 12 - On very large inputs, recursion or memory limits may be reached due to Python's recursion depth and slicing. 
 8 """ 13 """ 
 9 14 
 n 10 from typing import List,  An y n 15 from typing import List,  T y peVar 
 11 16 
 n n 17 T = TypeVar("T")  # Elements must support <= comparisons with each other 
 18   
 19 __all__ = ["merge_sort"] 
 20   
 12 def merge_sort(arr: List[ Any ]) -> List[ Any ]: 21 def merge_sort(arr: List[ T ]) -> List[ T ]: 
 13     """Sorts a list using the merge sort algorithm (returns a new sorted list). 22     """Return a new sorted list from the input list using merge sort. 
 14      23   
 24     The original input list is not modified. 
 25   
 15     Args: 26     Args: 
 n 16         arr: List of comparable elements. n 27         arr: List of  mutually  comparable elements. 
 17          28   
 18     Returns: 29     Returns: 
 19         New sorted list containing the same elements as arr. 30         New sorted list containing the same elements as arr. 
 n 20      n 
 21     Note: 
 22         The original list is not modified; a new sorted list is returned. 
 23         Uses list slicing, which copies data at each recursion level. 
 24         For large datasets, consider an in-place variant to reduce memory usage. 
 25         Merge sort is stable: equal elements retain their original order. 
 26         Very large lists may hit Python's maximum recursion depth. 
 27     """ 31     """ 
 28     if len(arr) <= 1: 32     if len(arr) <= 1: 
 29         return arr 33         return arr 
 30     mid = len(arr) // 2 34     mid = len(arr) // 2 
 31     left = merge_sort(arr[:mid]) 35     left = merge_sort(arr[:mid]) 
 32     right = merge_sort(arr[mid:]) 36     right = merge_sort(arr[mid:]) 
 n 33     return merge(left, right) n 37     return  _ merge(left, right) 
 34 38 
 n 35 def merge(left: List[ Any ], right: List[ Any ]) -> List[ Any ]: n 39 def  _ merge(left: List[ T ], right: List[ T ]) -> List[ T ]: 
 36     """Merge s  two sorted lists into one sorted list. 40     """Merge two sorted lists into one sorted list. 
 37      41   
 38     Preconditions: 42     Both `left` and `right` must be sorted lists of mutually comparable elements. 
 39         Both `left` and `right` must be sorted lists. 43   
 40      
 41     Args: 44     Args: 
 42         left: First sorted list. 45         left: First sorted list. 
 43         right: Second sorted list. 46         right: Second sorted list. 
 n 44          n 47   
 45     Returns: 48     Returns: 
 46         Merged sorted list containing all elements from left and right. 49         Merged sorted list containing all elements from left and right. 
 47     """ 50     """ 
 48     result = [] 51     result = [] 
 49     i = j = 0 52     i = j = 0 
 n 50     # Merge sort is stable; this comparison preserves order of equal elements. n 
 51     while i < len(left) and j < len(right): 53     while i < len(left) and j < len(right): 
 